In [1]:
from pulp import LpMaximize, LpProblem, LpVariable, lpSum
import pandas as pd
# from sklearn import LinearRegression

In [2]:
# Constants
position_limits = {
    "QB": 2,
    "RB": 2,
    "WR": 3,
    "TE": 1,
    "FLEX": 2,  # RB, WR, TE can qualify
    "K": 1,
    "DEF": 1
}
flex_positions = {"RB", "WR", "TE"}
balance_constant = 10  # Adjust this to balance points vs team odds

In [3]:
players = [
    {"name": "Mahomes", "team": "Chiefs", "position": "QB", "points": 18.4, "team_odds": 0.222},
    {"name": "Pacheco", "team": "Chiefs", "position": "RB", "points": 8.1, "team_odds": 0.222},
    {"name": "Hunt", "team": "Chiefs", "position": "RB", "points": 12, "team_odds": 0.222},
    {"name": "Worthy", "team": "Chiefs", "position": "WR", "points": 11.01, "team_odds": 0.222},
    {"name": "Hopkins", "team": "Chiefs", "position": "WR", "points": 9.19, "team_odds": 0.222},
    {"name": "Kelce", "team": "Chiefs", "position": "TE", "points": 12.02, "team_odds": 0.222},
    {"name": "Chiefs", "team": "Chiefs", "position": "DEF", "points": 6.53, "team_odds": 0.222},
    {"name": "Butker", "team": "Chiefs", "position": "K", "points": 7.23, "team_odds": 0.222},
    {"name": "Allen", "team": "Bills", "position": "QB", "points": 22.6, "team_odds": 0.143},
    {"name": "Cook", "team": "Bills", "position": "RB", "points": 16.7, "team_odds": 0.143},
    {"name": "Davis", "team": "Bills", "position": "RB", "points": 6.8, "team_odds": 0.143},
    {"name": "Shakir", "team": "Bills", "position": "WR", "points": 12.17, "team_odds": 0.143},
    {"name": "Cooper", "team": "Bills", "position": "WR", "points": 8.76, "team_odds": 0.143},
    {"name": "Kincaid", "team": "Bills", "position": "TE", "points": 7.75, "team_odds": 0.143},
    {"name": "Bills", "team": "Bills", "position": "DEF", "points": 8.59, "team_odds": 0.143},
    {"name": "Bass", "team": "Bills", "position": "K", "points": 8.0, "team_odds": 0.143},
    {"name": "Jackson", "team": "Ravens", "position": "QB", "points": 25.6, "team_odds": 0.143},
    {"name": "Henry", "team": "Ravens", "position": "RB", "points": 19.8, "team_odds": 0.143},
    {"name": "Hill", "team": "Ravens", "position": "RB", "points": 8.5, "team_odds": 0.143},
    {"name": "Flowers", "team": "Ravens", "position": "WR", "points": 12.32, "team_odds": 0.143},
    {"name": "Bateman", "team": "Ravens", "position": "WR", "points": 10.27, "team_odds": 0.143},
    {"name": "Andrews", "team": "Ravens", "position": "TE", "points": 11.11, "team_odds": 0.143},
    {"name": "Ravens", "team": "Ravens", "position": "DEF", "points": 7.59, "team_odds": 0.143},
    {"name": "Tucker", "team": "Ravens", "position": "K", "points": 7.82, "team_odds": 0.143},
    {"name": "Stroud", "team": "Texans", "position": "QB", "points": 13.7, "team_odds": 0.013},
    {"name": "Mixon", "team": "Texans", "position": "RB", "points": 17.2, "team_odds": 0.013},
    {"name": "Pierce", "team": "Texans", "position": "RB", "points": 4.0, "team_odds": 0.013},
    {"name": "Collins", "team": "Texans", "position": "WR", "points": 17.55, "team_odds": 0.013},
    {"name": "Metchie", "team": "Texans", "position": "WR", "points": 4.26, "team_odds": 0.013},
    {"name": "Schultz", "team": "Texans", "position": "TE", "points": 6.95, "team_odds": 0.013},
    {"name": "Texans", "team": "Texans", "position": "DEF", "points": 8.41, "team_odds": 0.013},
    {"name": "Fairbairn", "team": "Texans", "position": "K", "points": 9.65, "team_odds": 0.013},
    {"name": "Herbert", "team": "Chargers", "position": "QB", "points": 17, "team_odds": 0.038},
    {"name": "Dobbins", "team": "Chargers", "position": "RB", "points": 14.8, "team_odds": 0.038},
    {"name": "Edwards", "team": "Chargers", "position": "RB", "points": 5.08, "team_odds": 0.038},
    {"name": "McConkey", "team": "Chargers", "position": "WR", "points": 15.06, "team_odds": 0.038},
    {"name": "Johnston", "team": "Chargers", "position": "WR", "points": 11.65, "team_odds": 0.038},
    {"name": "Dissly", "team": "Chargers", "position": "TE", "points": 7.34, "team_odds": 0.038},
    {"name": "Chargers", "team": "Chargers", "position": "DEF", "points": 7.59, "team_odds": 0.038},
    {"name": "Dicker", "team": "Chargers", "position": "K", "points": 10.18, "team_odds": 0.038},
    {"name": "Wilson", "team": "Steelers", "position": "QB", "points": 16.2, "team_odds": 0.013},
    {"name": "Harris", "team": "Steelers", "position": "RB", "points": 12.0, "team_odds": 0.013},
    {"name": "Warren", "team": "Steelers", "position": "RB", "points": 8.3, "team_odds": 0.013},
    {"name": "Pickens", "team": "Steelers", "position": "WR", "points": 11.74, "team_odds": 0.013},
    {"name": "Jefferson", "team": "Steelers", "position": "WR", "points": 3.74, "team_odds": 0.013},
    {"name": "Freiermuth", "team": "Steelers", "position": "TE", "points": 9.9, "team_odds": 0.013},
    {"name": "Steelers", "team": "Steelers", "position": "DEF", "points": 9.29, "team_odds": 0.013},
    {"name": "Boswell", "team": "Steelers", "position": "K", "points": 11.06, "team_odds": 0.013},
    {"name": "Nix", "team": "Broncos", "position": "QB", "points": 19.4, "team_odds": 0.016},
    {"name": "McLaughlin", "team": "Broncos", "position": "RB", "points": 6.1, "team_odds": 0.016},
    {"name": "Estime", "team": "Broncos", "position": "RB", "points": 3.7, "team_odds": 0.016},
    {"name": "Sutton", "team": "Broncos", "position": "WR", "points": 14.14, "team_odds": 0.016},
    {"name": "Vele", "team": "Broncos", "position": "WR", "points": 8.19, "team_odds": 0.016},
    {"name": "Trautman", "team": "Broncos", "position": "TE", "points": 2.58, "team_odds": 0.016},
    {"name": "Broncos", "team": "Broncos", "position": "DEF", "points": 10.24, "team_odds": 0.016},
    {"name": "Lutz", "team": "Broncos", "position": "K", "points": 9.24, "team_odds": 0.016},
    {"name": "Goff", "team": "Lions", "position": "QB", "points": 19.8, "team_odds": 0.267},
    {"name": "Gibbs", "team": "Lions", "position": "RB", "points": 21.3, "team_odds": 0.267},
    {"name": "Montgomery", "team": "Lions", "position": "RB", "points": 15.8, "team_odds": 0.267},
    {"name": "St. Brown", "team": "Lions", "position": "WR", "points": 18.6, "team_odds": 0.267},
    {"name": "Williams", "team": "Lions", "position": "WR", "points": 14.15, "team_odds": 0.267},
    {"name": "LaPorta", "team": "Lions", "position": "TE", "points": 10.91, "team_odds": 0.267},
    {"name": "Lions", "team": "Lions", "position": "DEF", "points": 7.65, "team_odds": 0.267},
    {"name": "Bates", "team": "Lions", "position": "K", "points": 9.12, "team_odds": 0.267},
    {"name": "Hurts", "team": "Eagles", "position": "QB", "points": 21.3, "team_odds": 0.125},
    {"name": "Barkley", "team": "Eagles", "position": "RB", "points": 22.3, "team_odds": 0.125},
    {"name": "Gainwell", "team": "Eagles", "position": "RB", "points": 3.7, "team_odds": 0.125},
    {"name": "Brown", "team": "Eagles", "position": "WR", "points": 16.68, "team_odds": 0.125},
    {"name": "Smith", "team": "Eagles", "position": "WR", "points": 15.34, "team_odds": 0.125},
    {"name": "Goedert", "team": "Eagles", "position": "TE", "points": 10.36, "team_odds": 0.125},
    {"name": "Eagles", "team": "Eagles", "position": "DEF", "points": 8.59, "team_odds": 0.125},
    {"name": "Elliott", "team": "Eagles", "position": "K", "points": 7.76, "team_odds": 0.125},
    {"name": "Mayfield", "team": "Buccaneers", "position": "QB", "points": 22.5, "team_odds": 0.032},
    {"name": "Irving", "team": "Buccaneers", "position": "RB", "points": 14.4, "team_odds": 0.032},
    {"name": "White", "team": "Buccaneers", "position": "RB", "points": 12.5, "team_odds": 0.032},
    {"name": "Evans", "team": "Buccaneers", "position": "WR", "points": 17.17, "team_odds": 0.032},
    {"name": "McMillan", "team": "Buccaneers", "position": "WR", "points": 10.42, "team_odds": 0.032},
    {"name": "Otton", "team": "Buccaneers", "position": "TE", "points": 10.04, "team_odds": 0.032},
    {"name": "Buccaneers", "team": "Buccaneers", "position": "DEF", "points": 6.59, "team_odds": 0.032},
    {"name": "McLaughlin", "team": "Buccaneers", "position": "K", "points": 9.41, "team_odds": 0.032},
    {"name": "Stafford", "team": "Rams", "position": "QB", "points": 13.9, "team_odds": 0.028},
    {"name": "Williams", "team": "Rams", "position": "RB", "points": 17, "team_odds": 0.028},
    {"name": "Rivers", "team": "Rams", "position": "RB", "points": 1.08, "team_odds": 0.028},
    {"name": "Nacua", "team": "Rams", "position": "WR", "points": 18.78, "team_odds": 0.028},
    {"name": "Kupp", "team": "Rams", "position": "WR", "points": 14.58, "team_odds": 0.028},
    {"name": "Higbee", "team": "Rams", "position": "TE", "points": 8.87, "team_odds": 0.028},
    {"name": "Rams", "team": "Rams", "position": "DEF", "points": 7.24, "team_odds": 0.028},
    {"name": "Karty", "team": "Rams", "position": "K", "points": 7.41, "team_odds": 0.028},
    {"name": "Darnold", "team": "Vikings", "position": "QB", "points": 18.8, "team_odds": 0.059},
    {"name": "Jones", "team": "Vikings", "position": "RB", "points": 14.2, "team_odds": 0.059},
    {"name": "Akers", "team": "Vikings", "position": "RB", "points": 5.5, "team_odds": 0.059},
    {"name": "Jefferson", "team": "Vikings", "position": "WR", "points": 18.68, "team_odds": 0.059},
    {"name": "Addison", "team": "Vikings", "position": "WR", "points": 14.17, "team_odds": 0.059},
    {"name": "Hockenson", "team": "Vikings", "position": "TE", "points": 8.65, "team_odds": 0.059},
    {"name": "Vikings", "team": "Vikings", "position": "DEF", "points": 9.94, "team_odds": 0.059},
    {"name": "Reichard", "team": "Vikings", "position": "K", "points": 9.62, "team_odds": 0.059},
    {"name": "Daniels", "team": "Commanders", "position": "QB", "points": 21.5, "team_odds": 0.024},
    {"name": "Robinson Jr", "team": "Commanders", "position": "RB", "points": 11.4, "team_odds": 0.024},
    {"name": "Ekeler", "team": "Commanders", "position": "RB", "points": 11.0, "team_odds": 0.024},
    {"name": "McLaurin", "team": "Commanders", "position": "WR", "points": 15.75, "team_odds": 0.024},
    {"name": "Zaccheaus", "team": "Commanders", "position": "WR", "points": 6.61, "team_odds": 0.024},
    {"name": "Ertz", "team": "Commanders", "position": "TE", "points": 10.41, "team_odds": 0.024},
    {"name": "Commanders", "team": "Commanders", "position": "DEF", "points": 5.76, "team_odds": 0.024},
    {"name": "Gonzalez", "team": "Commanders", "position": "K", "points": 6.0, "team_odds": 0.024},
    {"name": "Love", "team": "Packers", "position": "QB", "points": 16.3, "team_odds": 0.053},
    {"name": "Jacobs", "team": "Packers", "position": "RB", "points": 17.2, "team_odds": 0.053},
    {"name": "Wilson", "team": "Packers", "position": "RB", "points": 5.65, "team_odds": 0.053},
    {"name": "Reed", "team": "Packers", "position": "WR", "points": 11.59, "team_odds": 0.053},
    {"name": "Doubs", "team": "Packers", "position": "WR", "points": 10.16, "team_odds": 0.053},
    {"name": "Kraft", "team": "Packers", "position": "TE", "points": 9.61, "team_odds": 0.053},
    {"name": "Packers", "team": "Packers", "position": "DEF", "points": 9.29, "team_odds": 0.053},
    {"name": "McManus", "team": "Packers", "position": "K", "points": 8.91, "team_odds": 0.053},
]

In [ ]:
# Create problem
problem = LpProblem("PlayoffFantasyTeamSelection", LpMaximize)

# Variables
variables = {p["name"]: LpVariable(p["name"], 0, 1, cat="Binary") for p in players}

# Objective function: Maximize points + weighted team odds
problem += lpSum([
    (player["points"] + balance_constant * player["team_odds"]) * variables[player["name"]]
    for player in players
])

# Constraints
# 1. Position limits
for position, limit in position_limits.items():
    if position == "FLEX":
        problem += lpSum([
            variables[player["name"]] for player in players if player["position"] in flex_positions
        ]) <= limit
    else:
        problem += lpSum([
            variables[player["name"]] for player in players if player["position"] == position
        ]) == limit

# 2. One player per team
# teams = set(player["team"] for player in players)
# for team in teams:
#     problem += lpSum([
#         variables[player["name"]] for player in players if player["team"] == team
#     ]) <= 1
max_players_per_team = 1
teams = set(player['team'] for player in players)
for team in teams:
    problem += lpSum(variables[player['name']] for player in players if player['team'] == team) <= max_players_per_team

# Solve the problem
problem.solve()

# Display selected team
selected_team = [player["name"] for player in players if variables[player["name"]].value() == 1]
print("Selected Team:", selected_team)

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/mtreigelman/opt/anaconda3/lib/python3.9/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/mf/tvg4nmds1zs6fkm_hx2t_dhh0000gn/T/c68d4fee62f040c4baeca9c099116b77-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/mf/tvg4nmds1zs6fkm_hx2t_dhh0000gn/T/c68d4fee62f040c4baeca9c099116b77-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 26 COLUMNS
At line 642 RHS
At line 664 BOUNDS
At line 773 ENDATA
Problem MODEL has 21 rows, 108 columns and 291 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Problem is infeasible - 0.00 seconds
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.00   (Wallclock seconds):       0.00

Selected Team: ['Kelce', 'Allen', 'Jackson', 'Gibbs', 'St. Brown', 'Bates', 'Brown', 'Nacua', 'Vikings']


In [ ]:
# Create the problem
prob = LpProblem("Fantasy_Team_Selection", LpMaximize)

# Decision variables
player_vars = {player['name']: LpVariable(player['name'], 0, 1, cat='Binary') for player in players}

# Objective function: Maximize points + weighted team odds
problem += lpSum([
    (player["points"] + BALANCE_CONSTANT * player["team_odds"]) * variables[player["name"]]
    for player in players
])

# Position constraints
prob += lpSum(player_vars[player['name']] for player in players if player['position'] == 'QB') == 2
prob += lpSum(player_vars[player['name']] for player in players if player['position'] == 'RB') == 2
prob += lpSum(player_vars[player['name']] for player in players if player['position'] == 'WR') == 3
prob += lpSum(player_vars[player['name']] for player in players if player['position'] == 'TE') == 1
prob += lpSum(player_vars[player['name']] for player in players if player['position'] == 'FLEX') == 2
prob += lpSum(player_vars[player['name']] for player in players if player['position'] == 'DEF') == 1
prob += lpSum(player_vars[player['name']] for player in players if player['position'] == 'K') == 1

for position, limit in POSITION_LIMITS.items():
    if position == "FLEX":
        problem += lpSum([
            variables[player["name"]] for player in players if player["position"] in FLEX_POSITIONS
        ]) <= limit
    else:
        problem += lpSum([
            variables[player["name"]] for player in players if player["position"] == position
        ]) == limit

max_players_per_team = 1
teams = set(player['team'] for player in players)
for team in teams:
    prob += lpSum(player_vars[player['name']] for player in players if player['team'] == team) <= max_players_per_team

model += lpSum(player_vars[player["name"]] for player in players) == 12, "TotalPlayers"

prob.solve()

selected_players = [player['name'] for player in players if player_vars[player['name']].value() == 1]
print("Selected Players:", selected_players)

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/mtreigelman/opt/anaconda3/lib/python3.9/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/mf/tvg4nmds1zs6fkm_hx2t_dhh0000gn/T/913475227db1433bb20ea4f09a248faf-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/mf/tvg4nmds1zs6fkm_hx2t_dhh0000gn/T/913475227db1433bb20ea4f09a248faf-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 34 COLUMNS
At line 869 RHS
At line 899 BOUNDS
At line 1008 ENDATA
Problem MODEL has 29 rows, 108 columns and 510 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Problem is infeasible - 0.00 seconds
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.00   (Wallclock seconds):       0.00

Selected Players: ['Allen', 'Jackson', 'Broncos', 'Gibbs', 'Lions', 'Bates', 'Barkley', 'Vikings']


In [7]:
# chat GPT
# Data preparation
df = pd.DataFrame(players)
df['weighted_score'] = df['points'] + balance_constant * df['team_odds']

# Create problem
problem = LpProblem("Optimal_Lineup", LpMaximize)

# Variables
player_vars = {i: LpVariable(f"player_{i}", cat="Binary") for i in df.index}

# Objective: Maximize the weighted score
problem += lpSum(player_vars[i] * df.loc[i, 'weighted_score'] for i in df.index)

# Constraints
# Only one player per team
for team in df['team'].unique():
    team_indices = df[df['team'] == team].index
    problem += lpSum(player_vars[i] for i in team_indices) <= 1

# Positional constraints
for position, limit in position_limits.items():
    if position == "FLEX":
        flex_indices = df[df['position'].isin(flex_positions)].index
        problem += lpSum(player_vars[i] for i in flex_indices) == limit
    else:
        pos_indices = df[df['position'] == position].index
        problem += lpSum(player_vars[i] for i in pos_indices) == limit

# Solve the problem
problem.solve()

# Extract selected players
selected_players = [df.loc[i, 'name'] for i in df.index if player_vars[i].value() == 1]
selected_positions = [df.loc[i, 'position'] for i in df.index if player_vars[i].value() == 1]
selected_teams = [df.loc[i, 'team'] for i in df.index if player_vars[i].value() == 1]

# Output results
print("Selected Players:")
for player, position, team in zip(selected_players, selected_positions, selected_teams):
    print(f"{player} ({position}) from {team}")

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /Users/mtreigelman/opt/anaconda3/lib/python3.9/site-packages/pulp/solverdir/cbc/osx/64/cbc /var/folders/mf/tvg4nmds1zs6fkm_hx2t_dhh0000gn/T/07836875de9649ad9e1646ad2d61b3ae-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/mf/tvg4nmds1zs6fkm_hx2t_dhh0000gn/T/07836875de9649ad9e1646ad2d61b3ae-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 26 COLUMNS
At line 657 RHS
At line 679 BOUNDS
At line 792 ENDATA
Problem MODEL has 21 rows, 112 columns and 294 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Problem is infeasible - 0.00 seconds
Option for printingOptions changed from normal to all
Total time (CPU seconds):       0.00   (Wallclock seconds):       0.01

Selected Players:
Allen (QB) from Bills
Jackson (QB) from Ravens
St. Brown (WR) from Lions
Bates (K) from Lions
Nacua (WR) from Rams
J